# AgriConnect — Crop Mandi Price Prediction Pipeline

This notebook demonstrates training an **XGBoost / Random Forest Regressor** on historical mandi market prices (from the Government of India Agmarknet data portal).

### Features:
- `crop`: Target produce (Tomato, Potato, Onion, Wheat, etc.)
- `district` & `state`: Geographic location
- `month` & `season`: Kharif, Rabi, Zaid cyclical factors
- `lag_1_price`, `lag_2_price`, `lag_3_price`: Rolling historical 3-month prices
- `rainfall_deviation`: Weather indicator (optional)

### Outputs:
- 3-month forward price forecast with low-band and high-band intervals
- Export to `../app/ml/artifacts/v1/price_model.pkl`

In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

print("Libraries loaded successfully.")

## 1. Synthetic / Historical Dataset Generation

In [ ]:
np.random.seed(42)
n_samples = 5000

crops = ['Tomato', 'Potato', 'Onion', 'Wheat', 'Rice', 'Maize', 'Grape']
districts = ['Nashik', 'Pune', 'Solapur', 'Nagpur', 'Kolhapur', 'Indore', 'Bhopal']

data = {
    'crop': np.random.choice(crops, n_samples),
    'district': np.random.choice(districts, n_samples),
    'month': np.random.randint(1, 13, n_samples),
    'lag_1_price': np.random.uniform(15, 80, n_samples),
    'lag_2_price': np.random.uniform(15, 80, n_samples),
    'lag_3_price': np.random.uniform(15, 80, n_samples),
}

df = pd.DataFrame(data)
# Target price = weighted lag + seasonality + noise
df['target_price'] = (
    0.5 * df['lag_1_price'] + 
    0.3 * df['lag_2_price'] + 
    0.2 * df['lag_3_price'] + 
    np.sin(df['month'] / 12 * 2 * np.pi) * 8 +
    np.random.normal(0, 3, n_samples)
).round(2)

print("Sample training data:")
print(df.head())

## 2. Feature Encoding & Split

In [ ]:
df_encoded = pd.get_dummies(df, columns=['crop', 'district'], drop_first=True)
X = df_encoded.drop(columns=['target_price'])
y = df_encoded['target_price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

## 3. Train Random Forest Regressor

In [ ]:
model = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)
r2 = r2_score(y_test, preds)

print(f"Test MAE: {mae:.2f} INR/kg")
print(f"R2 Score: {r2:.4f}")

## 4. Export Artifact

In [ ]:
export_dir = '../app/ml/artifacts/v1'
os.makedirs(export_dir, exist_ok=True)
output_pkl = os.path.join(export_dir, 'price_model.pkl')

with open(output_pkl, 'wb') as f:
    pickle.dump(model, f)

print(f"Saved trained price regressor to {output_pkl}")